In [1]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.tools import tool
import re
from typing import List,Dict,Union,Optional,Any

In [2]:
import numpy as np
import pandas as pd
import matplotlib
import seaborn
import langchain
import sklearn
import glob


In [3]:
import urllib.request

urls = [
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/N0CceRlquaf9q85PK759WQ/regression-dataset.csv",
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7J73m6Nsz-vmojwab91gMA/classification-dataset.csv",
]

for url in urls:
    filename = os.path.basename(url)
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists")

regression-dataset.csv already exists
classification-dataset.csv already exists


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [5]:
@tool
def list_csv_files() -> Optional[List[str]]:
    """
    List all CSV file names in the local directory
    
    Returns:
    A list containing CSV file name 
    if no csv files are found, return None.
    """
    csv_files=glob.glob(os.path.join(os.getcwd(),"*.csv"))
    if not csv_files:
        return None
    return [os.path.basename(file) for file in csv_files]

In [6]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

Tool Name: list_csv_files
Tool Description: List all CSV file names in the local directory

Returns:
A list containing CSV file name 
if no csv files are found, return None.
Tool Arguments: {}


#### Dataset Caching Tool

In [7]:
DATAFRAME_CACHE={}

@tool
def preload_datasets(paths: List[str]) -> str:
    """
    Loads CSV files into a global cache if not already loaded.
    
    This functions helps to efficiently manage datasets by loading them once
    and storing them in memory for future use. without caching, you would waste
    tokens describing datasets content repeatedly in agent response
    
    Args:
        paths: A list of file path to csv files.
    Returns:
        A message summarizing which datasets were loaded or already cached
    """
    
    loaded=[]
    cached=[]
    
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] =pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)
    return(
        f'Loaded datasets: {loaded}\n'
        f'Already cached:{cached}'
    )
            

##### Visualize Classifications datasets

In [41]:
from typing import Dict, List
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from langchain.tools import tool
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


@tool
def visualize_classification_dataset(
    file_name: str,
    target_column: str
) -> Dict[str, List[str]]:
    """
    Train a classifier and generate visualization plots.

    Args:
        file_name (str): Dataset name or path.
        target_column (str): Classification target column.

    Returns:
        Dict[str, List[str]]:
            Paths to generated visualization files.
    """

    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": [f"DataFrame '{file_name}' not found."]}
        except Exception as e:
            return {"error": [str(e)]}

    df = DATAFRAME_CACHE[file_name]

    if target_column not in df.columns:
        return {"error": [f"Target column '{target_column}' not found."]}

    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    image_paths = []

    # Confusion Matrix
    plt.figure()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.savefig("confusion_matrix.png")
    plt.close()
    image_paths.append("confusion_matrix.png")

    # Feature Importance
    plt.figure(figsize=(8,5))
    importance = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    sns.barplot(
        x=importance.values,
        y=importance.index
    )

    plt.title("Feature Importance")
    plt.tight_layout()
    plt.savefig("classification_feature_importance.png")
    plt.close()
    image_paths.append("classification_feature_importance.png")

    # Class Distribution
    plt.figure(figsize=(6,4))
    sns.countplot(x=y)
    plt.tight_layout()
    plt.savefig("class_distribution.png")
    plt.close()
    image_paths.append("class_distribution.png")

    return image_paths

#### visualization Regressions datasets

In [43]:
from sklearn.metrics import PredictionErrorDisplay
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


from typing import List
from matplotlib.figure import Figure

@tool
def visualize_regression_dataset(
    file_name: str,
    target_column: str
) -> List[Figure]:
    """
    Train a regression model and generate visualization plots.

    Args:
        file_name (str): Dataset name or path.
        target_column (str): Regression target column.

    Returns:
        List[Figure]:
            List of matplotlib figures.
    """

    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return []
        except Exception:
            return []

    df = DATAFRAME_CACHE[file_name]

    if target_column not in df.columns:
        return []

    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestRegressor()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    figures = []

    # Actual vs Predicted
    fig1, ax1 = plt.subplots(figsize=(6, 6))

    ax1.scatter(y_test, y_pred)
    ax1.set_xlabel("Actual")
    ax1.set_ylabel("Predicted")
    ax1.set_title("Actual vs Predicted")

    figures.append(fig1)

    # Residual Plot
    residuals = y_test - y_pred

    fig2, ax2 = plt.subplots(figsize=(6, 5))

    sns.scatterplot(
        x=y_pred,
        y=residuals,
        ax=ax2
    )

    ax2.axhline(0, color="red")
    ax2.set_xlabel("Predicted")
    ax2.set_ylabel("Residuals")
    ax2.set_title("Residual Plot")

    figures.append(fig2)

    # Feature Importance
    importance = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    fig3, ax3 = plt.subplots(figsize=(8, 5))

    sns.barplot(
        x=importance.values,
        y=importance.index,
        ax=ax3
    )

    ax3.set_title("Feature Importance")

    figures.append(fig3)

    return figures

#### Summarization tool

In [8]:
@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str,Any]]:
    """
    Analyze multiple CSV files and return metadata summaries for each.
    
    Args:
        dataset_paths(List[str]):
            A list of file paths to CSV datasets
    Returns:
        List[Dict[str,Any]]:
            A list of summaries, one per dataset, each containing:
                -"file_name": The path of the dataset file.
                -"column_name": A list of column names in the dataset.
                -"data_types": A dictionary mapping columns names to their data
                                types(as string).
    """
    summaries=[]
    
    for path in dataset_paths:
        # Load and cache the datasets if not already cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] =pd.read_csv(path)
        
        df=DATAFRAME_CACHE[path]
        
        # Build summary
        summary={
            'file_name':path,
            'column_names':df.columns.tolist(),
            'data_types':df.dtypes.astype(str).to_dict()
        }
        summaries.append(summary)
    return summaries

#### DataFrame method execution tool

In [9]:
@tool
def call_dataframe_method(file_name: str, method:str) -> str:
    """
    Execute a method on a DataFrame and return the result.
    This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe'.capitalize
    on a datasets that has already been loaded and cached using 'preload_datasets'
    
    Args:
        file_name(str): The path or name of the datasets in the global cache.
        method(str): the name of the method to call on the Dataframe. Only no-argument
        method are supported (e.g. 'head','describe','info').
    
    Returns:
        str: The output of the method as a formatted string, or an error message
            if the datasets is not found or the method is invalid
    
    Example:
        call_dataframe_method(file_name='data.csv',method='head')
    """
    
    # Try to get the DataFrame from the cache, or load it if not already cached
    
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name]=pd.read_csv(file_name)
        except FileNotFoundError:
            return f"Dataframe '{file_name}' not found in cache or in disk"
        
        except Exception as e:
            return f"Error loading '{file_name}':{str(e)}"
        
    df=DATAFRAME_CACHE[file_name]
    func=getattr(df,method,None)      
    if not callable(func):
        return f"'{method}' is not valid method of DataFrame"
    try:
        result=func()
        return str(result)
    except Exception as e:
        return f"error calling '{method}' on '{file_name}': {str(e)}"  

#### Model evaluation tools

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import accuracy_score,r2_score,mean_absolute_error,mean_squared_error

In [11]:
# Assume this global cache is shared
DATAFRAME_CACHE={}

@tool
def evaluate_classification_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a classifier on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the classification target.
    Returns:
        Dict[str, float]: A dictionary with the model's accuracy score.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return {"accuracy": acc}

@tool
def evaluate_regression_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a regression model on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the regression target.
    Returns:
        Dict[str, float]: A dictionary with R² score and Mean Squared Error.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    return {
        "r2_score": r2,
        "mean_squared_error": mse
    }

#### Agents

In [12]:
prompt=ChatPromptTemplate.from_messages([
    ('system',"you are a data science assistant. Use the available tools to anlayze CSV files "
     "Your job is to determine whether each datasets is for classification or regression, based on its structure"),
    
    ('user',"{input}"),
    ('placeholder','{agent_scratchpad}') #required for tool calling agents
])

#### Combined tools

In [32]:
tools=[list_csv_files, preload_datasets, get_dataset_summaries, call_dataframe_method, evaluate_classification_dataset, evaluate_regression_dataset,visualize_classification_dataset,visualize_regression_dataset]

#### Agent creations and limitations

In [33]:
from langchain_classic.agents import create_openai_tools_agent

In [34]:
agent = create_openai_tools_agent(llm, tools, prompt)

In [35]:
response = agent.invoke({
    "input": "Can you tell me about the dataset?",
    "intermediate_steps": []
})

In [36]:
# Get the first ToolAgentAction from the list
action = response[0]

# Print the key details
print("🧠 Agent decided to call a tool:")
print("Tool Name:", action.tool)
print("Tool Input:", action.tool_input)
print("Log:\n", action.log.strip())

🧠 Agent decided to call a tool:
Tool Name: list_csv_files
Tool Input: {}
Log:
 Invoking: `list_csv_files` with `{}`
responded: I'll help you analyze the available dataset(s). Let me first find what CSV files are available.


In [37]:
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent

In [38]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False,handle_parsing_errors=True)
agent_executor.agent.stream_runnable = False

In [45]:
import gradio as gr

import gradio as gr
import matplotlib.pyplot as plt


import gradio as gr
from matplotlib.figure import Figure

def chat_with_agent(message, history):

    result = agent_executor.invoke({"input": message})
    output = result["output"]

    if isinstance(output, list) and all(
        isinstance(fig, Figure) for fig in output
    ):
        return output

    return str(output)


demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="📊 Dataset Analysis Agent",
    description="Ask questions about your dataset"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
